# Train note_detector on Colab

Real note+octave arpeggio detection (upgrade from the already-trained
pitch-class MLP baked into the app). Uses synthetic MIDI-rendered arpeggios
— no dataset to download.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (free tier is fine).

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — go enable a GPU runtime")

## 1. Get the training code onto this Colab instance

Two options — pick one:

**Option A — clone your private repo** (replace `YOUR_GITHUB_TOKEN` with a
[fine-grained personal access token](https://github.com/settings/tokens)
scoped to just this repo; never paste a token into a notebook you'll share):

In [ ]:
# Option A: clone your private repo
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"  # replace before running
!git clone https://{GITHUB_TOKEN}@github.com/Zetthilly/Ichi.git
%cd Ichi/training/note_detector

**Option B — upload the folder directly** (skip the cell above if you use this instead):

In [ ]:
# Option B: manual upload — run this instead of the git clone above
from google.colab import files
import zipfile, io

print("Select the zipped training/note_detector folder (zip it locally first: cd training && zip -r note_detector.zip note_detector)")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
    z.extractall(".")
%cd note_detector

## 2. Install dependencies

In [ ]:
!pip install -q torch torchaudio pretty_midi soundfile numpy ai-edge-torch

## 3. Generate the synthetic training set

`examples_per_combo=20` -> 12 roots x 9 qualities x 5 directions x 20 ≈
10,800 examples. Raise this if training loss plateaus too early; each unit
of `examples_per_combo` adds a few minutes to this cell.

In [ ]:
!python generate_synthetic_arpeggios.py --output_dir data/arpeggios --examples_per_combo 20

## 4. Train

Watch `val_loss` — it should trend down and track train_loss without
diverging upward. `exact_frame_accuracy` will look modest even for a good
model (it requires all 48 note on/off states correct simultaneously, per
frame) — that's expected, not a bug.

In [ ]:
!python train.py --data_dir data/arpeggios --epochs 30 --batch_size 32 --output checkpoints/note_detector.pt

## 5. Export to TFLite

In [ ]:
!python export_to_tflite.py --checkpoint checkpoints/note_detector.pt --output note_detector.tflite

## 6. Download the result

Grabs the `.tflite` file plus the checkpoint (keep the checkpoint — you can
resume/fine-tune further later without redoing training from scratch).

In [ ]:
from google.colab import files
files.download("note_detector.tflite")
files.download("checkpoints/note_detector.pt")

## 7. Bring it into the app

1. Copy `note_detector.tflite` to `app/src/main/assets/models/` in the
   Android project.
2. Write a small `Interpreter`-based Kotlin wrapper in `core-audio`
   (`TrainedPitchClassifier.kt` and `TFLiteStemSeparator.kt` are both
   reference patterns for this) that:
   - Computes a log-mel spectrogram on-device matching `N_FFT`/`HOP_LENGTH`/
     `N_MELS` from `dataset.py` in this folder.
   - Feeds it through the interpreter — input shape
     `[1, 1, 128, CROP_FRAMES]`, output `[1, CROP_FRAMES, 48]` per-frame
     note-activation logits (apply sigmoid).
   - Maps output indices back to note names/octaves using `NOTE_MIN`/
     `NOTE_MAX` from `generate_synthetic_arpeggios.py` (MIDI 48-95, i.e. C3-B6).
3. Swap it in as the engine behind `HybridArpeggioNoteDetector` (same
   integration point Note-by-Note mode already uses).